In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from math import pi

#parametros
f=60
T=1 / f
w=2 * pi * f
A=1

alpha_g=0
alpha=(alpha_g*pi)/180

ciclos=4
muestreo=10000
t_total=ciclos*T
t=np.linspace(0, t_total, ciclos*muestreo)

#voltajes de linea
phi_g=[0, 120, 240]
phi=np.deg2rad(phi_g)
theta=w*t

vab=A*np.sin(theta+phi[0])
vba=-vab

vca=A*np.sin(theta+phi[1])
vac=-vca

vbc=A*np.sin(theta+phi[2])
vcb=-vbc

#RECTIFICACIoN UNA SOLA FASE:
#Busca indice de conducciin entre vab y vac (conduccion para vab)
idx_vab=np.argmax(theta>=np.deg2rad(60)+alpha)
idx_vac=np.argmax(theta>=2*np.deg2rad(60)+alpha)

#rectificacion de una fase
y_fase=vab[idx_vab:idx_vac]  #extrae la rectificacion de una fase
num_fases=round(muestreo/len(y_fase))  #numero de fases en un ciclo
puntos_unafase=len(y_fase)  #puntos de una sola fase

#CONSTRUCCIoN DE RECTIFICACIoN DE UN SOLO CICLO (PERIODO)
y_ciclo=np.zeros(muestreo)  #vector salida para un ciclo (periodo)
for k in range(1, num_fases + 1):
    inicio_fase = (k - 1) * puntos_unafase + idx_vab + 1
    fin_fase = k * puntos_unafase + idx_vab

    #si se sale del ciclo
    if fin_fase > muestreo:
        #corta ultimo pedazo de gráfica para no exceder
        diferencia_final = muestreo - inicio_fase
        y_ciclo[inicio_fase:] = y_fase[:diferencia_final]

        #pega ultimo pedazo de grafica excedido (del anterior) al inicio
        diferencia_inicial = fin_fase - muestreo + 1
        y_ciclo[:diferencia_inicial] = y_fase[diferencia_final:]

        #calcula fases intermedias no calculadas al inicio partiendo desde
        #ultimo pedazo añadido al principio
        diferencia_faltantes = idx_vab - diferencia_inicial
        ciclos_faltantes = round(diferencia_faltantes / puntos_unafase)
        for j in range(1, ciclos_faltantes + 1):
            inicio_fase_faltante = ((j - 1) * puntos_unafase) + diferencia_inicial 
            fin_fase_faltante = (j * puntos_unafase) + diferencia_inicial

            if fin_fase_faltante >= idx_vab:
                dif_ultimo_faltante_vab = idx_vab - inicio_fase_faltante
                y_ciclo[inicio_fase_faltante:idx_vab] = y_fase[:dif_ultimo_faltante_vab]
            else:
                y_ciclo[inicio_fase_faltante:fin_fase_faltante] = y_fase
        break
    else:
        y_ciclo[inicio_fase:fin_fase] = y_fase[:fin_fase - inicio_fase]

#CONSTRUCCIoN DE RECTIFICACIoN PARA VARIOS CICLOS (SEÑAL COMPLETA)
y_out = np.zeros(len(theta))

#une pedazos de cada ciclo en señal completa
for i in range(1, ciclos + 1):
    y_out[(i - 1) * len(y_ciclo):i * len(y_ciclo)] = y_ciclo

#el inicio es 0 hasta angulo de disparo
y_out[:idx_vab] = 0

#filtrado de la señal (por ahora no se realiza filtrado, pero se puede agregar si es necesario)

#GRAFICACION
def plot_rectified(alpha_g):
    #actualiza alpha
    alpha = (alpha_g * pi) / 180
    
    #recalculo de los voltajes y la salida
    vab = A * np.sin(theta + phi[0])
    vba = -vab
    vca = A * np.sin(theta + phi[1])
    vac = -vca
    vbc = A * np.sin(theta + phi[2])
    vcb = -vbc

    #rectificacion
    idx_vab = np.argmax(theta >= np.deg2rad(60) + alpha)
    idx_vac = np.argmax(theta >= 2 * np.deg2rad(60) + alpha)
    y_fase = vab[idx_vab:idx_vac]
    num_fases = round(muestreo / len(y_fase))
    puntos_unafase = len(y_fase)
    
    y_ciclo = np.zeros(muestreo)
    for k in range(1, num_fases + 1):
        inicio_fase = (k - 1) * puntos_unafase + idx_vab
        fin_fase = k * puntos_unafase + idx_vab
        if fin_fase > muestreo:
            diferencia_final = muestreo - inicio_fase
            y_ciclo[inicio_fase:] = y_fase[:diferencia_final]
            diferencia_inicial = fin_fase - muestreo
            y_ciclo[:diferencia_inicial] = y_fase[diferencia_final:]
            diferencia_faltantes = idx_vab - diferencia_inicial
            ciclos_faltantes = round(diferencia_faltantes / puntos_unafase)
            for j in range(1, ciclos_faltantes + 1):
                inicio_fase_faltante = ((j - 1) * puntos_unafase) + diferencia_inicial
                fin_fase_faltante = (j * puntos_unafase) + diferencia_inicial
                if fin_fase_faltante >= idx_vab:
                    dif_ultimo_faltante_vab = idx_vab - inicio_fase_faltante
                    y_ciclo[inicio_fase_faltante:idx_vab] = y_fase[:dif_ultimo_faltante_vab]
                else:
                    y_ciclo[inicio_fase_faltante:fin_fase_faltante] = y_fase
            break
        else:
            y_ciclo[inicio_fase:fin_fase] = y_fase[:fin_fase - inicio_fase]
    
    y_out = np.zeros(len(theta))
    for i in range(1, ciclos + 1):
        y_out[(i - 1) * len(y_ciclo):i * len(y_ciclo)] = y_ciclo
    
    y_out[:idx_vab] = 0
    
    #gráfica
    plt.figure(figsize=(10, 6))
    plt.plot(theta, vab, 'b', label='Vab')
    plt.plot(theta, vba, 'b--', label='Vba')
    plt.plot(theta, vca, 'g', label='Vac')
    plt.plot(theta, vcb, 'g--', label='Vcb')
    plt.plot(theta, vbc, 'r--', label='Vbc')
    plt.plot(theta, vac, 'r', label='Vca')
    plt.plot(theta, y_out, 'k', linewidth=2, label='Rectificada')
    plt.grid(True)
    plt.legend(loc='best')
    plt.xticks([0, pi, 2*pi, 3*pi, 4*pi, 5*pi, 6*pi, 7*pi, 8*pi], 
               ['0', r'$\pi$', r'$2\pi$', r'$3\pi$', r'$4\pi$', r'$5\pi$', r'$6\pi$', r'$7\pi$', r'$8\pi$'])
    plt.show()

#interactuar con el parámetro alpha_g
interact(plot_rectified, alpha_g=(0, 180, 1))

interactive(children=(IntSlider(value=90, description='alpha_g', max=180), Output()), _dom_classes=('widget-in…

<function __main__.plot_rectified(alpha_g)>